In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

In [ ]:
import alpine
import torch
from torch.optim.lr_scheduler import StepLR
import numpy as np


from matplotlib import pyplot as plt
from functools import partial
from torchmetrics.image import PeakSignalNoiseRatio
from torchmetrics import MetricTracker
import skimage.transform

from alpine.models import FourierReparameterization, Siren
from alpine.models.utils import get_coords_spatial



In [ ]:
epochs = 1000
H, W = 256, 256
nonlinearity = 'sine' # sine or relu -> use sine to compare to siren
scale = 0.05 if nonlinearity == 'relu' else 0.01

In [ ]:
# Prepare models

fourier_model = FourierReparameterization(in_features=2,
                                          out_features=3,
                                          hidden_features=256,
                                          hidden_layers=5,
                                          num_phases=32,
                                          num_frequencies=128,
                                          scaling_factor=scale,
                                          nonlinearity=nonlinearity,
                                          omegas=[30.0] if nonlinearity == 'sine' else None).cuda()
original_model = Siren(in_features=2,
                       out_features=3,
                       hidden_features=256,
                       hidden_layers=5,
                       omegas=[30.0],
                       outermost_linear=True).cuda()



In [ ]:
## Prepare data
coords = get_coords_spatial(256, 256).cuda()[None,...]
gt_img = skimage.transform.resize(skimage.data.astronaut(), (256,256))
gt = torch.from_numpy(gt_img).float().cuda()[None,...]


In [ ]:
results = dict()
for model, name in zip([fourier_model, original_model], ['Siren+FR', 'Siren']):
    scheduler=partial(StepLR, step_size=2500, gamma=0.1)
    model.compile(learning_rate=1e-4, scheduler=scheduler)
    outputs = model.fit_signal(input=coords, signal=gt, n_iters=epochs, enable_tqdm=True, return_features=True,
                                    metric_trackers = {'psnr': MetricTracker(PeakSignalNoiseRatio(1).to('cuda')),
                                                       'ssim' : MetricTracker(alpine.metrics.SSIM(signal_shape=(H,W)).to('cuda'))
                                                       },
                                    track_loss_history=True)
    results[name] = outputs

In [ ]:
fig, ax = plt.subplots(1,3,figsize=(15,5))
for k in results.keys():
    ax[0].plot(results[k]['loss_history'])
    ax[0].set_title('Loss')
    ax[1].plot(results[k]['metrics']['psnr'].numpy())
    ax[1].set_title('PSNR')
    ax[2].plot(results[k]['metrics']['ssim'].numpy())
    ax[2].set_title('SSIM')
ax[2].legend(list(results.keys()), loc='lower right')
plt.show()

In [ ]:
fig, ax = plt.subplots(2,3,figsize=(15,10))
ax[0,0].imshow(gt.cpu().numpy().reshape((256,256,3)))
ax[0,0].set_title('GT')

ax[1,0].imshow(np.zeros_like(gt[0].cpu()))
ax[1,0].set_title('Error')

for en, k in enumerate(results.keys()):
    ax[0,en+1].imshow(results[k]['output'].detach().cpu().numpy().reshape((256,256,3)))
    ax[1,en+1].imshow((gt - results[k]['output']).detach().cpu().numpy().reshape((256,256,3)))
    ax[0,en+1].set_title(f'{k}')
    ax[1,en+1].set_title(f'{k} - Error')

plt.show()